# Your own style

A style is a `dict`: change one, or write one from scratch. Here, Positron with blue water, and a layer added for green parks (in OpenMapTiles, city parks are `grass` in the `landcover` layer).

In [ ]:
%pip install -q basemaplibre geopandas

In [ ]:
import geopandas
import httpx
import matplotlib.pyplot as plt

import basemaplibre as bml

In [ ]:
def around(lon, lat, half_width):
    """Axes limits of a square around a point, half_width Web Mercator meters to each side."""
    x, y = geopandas.GeoSeries.from_xy([lon], [lat], crs=4326).to_crs(3857).iloc[0].coords[0]
    return x - half_width, x + half_width, y - half_width, y + half_width

In [ ]:
style = httpx.get(bml.STYLES["OpenFreeMap.Positron"]).json()
for layer in style["layers"]:
    if layer["id"] == "water":
        layer["paint"]["fill-color"] = "#9ec5e8"
grass = {
    "id": "grass",
    "type": "fill",
    "source": "openmaptiles",
    "source-layer": "landcover",
    "filter": ["==", ["get", "class"], "grass"],
    "paint": {"fill-color": "#bfe0b0"},
}
style["layers"].insert([layer["id"] for layer in style["layers"]].index("water"), grass)

fig, ax = plt.subplots(figsize=(5, 5))
ax.axis(around(-73.9665, 40.7812, 3200))
ax.set_aspect("equal")
ax.set_axis_off()
bml.add_basemap(ax, source=style)